<a href="https://colab.research.google.com/github/boss-defender/Born-Baby-Ai/blob/main/Just_born_Baby_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Cell 1
!pip -q install transformers datasets sentencepiece

In [1]:
# @title Cell 2
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [2]:
# @title Cell 3
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary:", tokenizer.vocab_size)

In [ ]:
# @title Cell 4
dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train"
)

dataset = dataset.shuffle(seed=42).select(range(10_000))

print(dataset)
print(dataset[0]["text"])

In [ ]:
# @title Cell 5
MAX_LENGTH = 128

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    return {
        "input_ids": tokens["input_ids"]
    }


dataset = dataset.map(
    tokenize,
    remove_columns=["text"]
)

dataset.set_format(
    type="torch",
    columns=["input_ids"]
)

print(dataset[0]["input_ids"].shape)

In [ ]:
# @title Cell 6
class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x

In [ ]:
# # @title cell 6 : Rotary positional embeddings
def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        (-x2, x1),
        dim=-1
    )


def apply_rope(q, k):

    T = q.shape[-2]
    D = q.shape[-1]

    position = torch.arange(
        T,
        device=q.device
    ).float()

    freq = 1.0 / (
        10000 ** (
            torch.arange(
                0,
                D,
                2,
                device=q.device
            ).float() / D
        )
    )

    angles = torch.outer(
        position,
        freq
    )

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    cos = torch.repeat_interleave(
        cos,
        2,
        dim=-1
    )

    sin = torch.repeat_interleave(
        sin,
        2,
        dim=-1
    )

    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]

    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin

    return q, k

In [ ]:
# @title Cell 7
class GQA(nn.Module):

    def __init__(
        self,
        dim,
        n_heads=8,
        n_kv_heads=2
    ):

        super().__init__()

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            dim,
            dim,
            bias=False
        )


    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = apply_rope(q, k)

        # Repeat KV heads
        repeat = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat,
            dim=1
        )

        v = v.repeat_interleave(
            repeat,
            dim=1
        )

        # Causal attention
        scores = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        mask = torch.triu(
            torch.ones(
                T,
                T,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        scores = scores.masked_fill(
            mask,
            float("-inf")
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = weights @ v

        out = out.transpose(
            1,
            2
        ).contiguous()

        out = out.view(
            B,
            T,
            C
        )

        return self.out_proj(out)

In [ ]:
# @title Cell 8
class SwiGLU(nn.Module):

    def __init__(self, dim, hidden_dim):

        super().__init__()

        self.gate = nn.Linear(
            dim,
            hidden_dim,
            bias=False
        )

        self.up = nn.Linear(
            dim,
            hidden_dim,
            bias=False
        )

        self.down = nn.Linear(
            hidden_dim,
            dim,
            bias=False
        )


    def forward(self, x):

        return self.down(
            F.silu(self.gate(x))
            * self.up(x)
        )

In [ ]:
# @title Cell 9
class MoE(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
        n_experts=4,
        top_k=2
    ):

        super().__init__()

        self.n_experts = n_experts
        self.top_k = top_k

        self.router = nn.Linear(
            dim,
            n_experts,
            bias=False
        )

        self.experts = nn.ModuleList([
            SwiGLU(
                dim,
                hidden_dim
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):

        B, T, C = x.shape

        router_logits = self.router(x)

        weights = F.softmax(
            router_logits,
            dim=-1
        )

        top_weights, top_ids = torch.topk(
            weights,
            self.top_k,
            dim=-1
        )

        top_weights = (
            top_weights /
            top_weights.sum(
                dim=-1,
                keepdim=True
            )
        )

        output = torch.zeros_like(x)

        for expert_id, expert in enumerate(
            self.experts
        ):

            mask = (
                top_ids == expert_id
            )

            if not mask.any():
                continue

            positions = mask.nonzero(
                as_tuple=False
            )

            token_indices = positions[:, 0]
            time_indices = positions[:, 1]
            choice_indices = positions[:, 2]

            selected = x[
                token_indices,
                time_indices
            ]

            expert_output = expert(
                selected
            )

            expert_weight = top_weights[
                token_indices,
                time_indices,
                choice_indices
            ]

            output[
                token_indices,
                time_indices
            ] += (
                expert_output *
                expert_weight.unsqueeze(-1)
            )

        return output

In [ ]:
# @title Cell 10
class Block(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.attn = GQA(
            dim,
            n_heads=8,
            n_kv_heads=2
        )

        self.norm2 = RMSNorm(dim)

        self.moe = MoE(
            dim=dim,
            hidden_dim=dim * 2,
            n_experts=4,
            top_k=2
        )

    def forward(self, x):

        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.moe(
            self.norm2(x)
        )

        return x

In [ ]:
# @title Cell 11
class BabyDeepSeek(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim=256,
        layers=6
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.blocks = nn.ModuleList([
            Block(dim)
            for _ in range(layers)
        ])

        self.norm = RMSNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

        # Tie input/output embeddings
        self.lm_head.weight = (
            self.token_embedding.weight
        )


    def forward(
        self,
        input_ids,
        targets=None
    ):

        x = self.token_embedding(
            input_ids
        )

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1)
            )

        return logits, loss

In [ ]:
# @title Cell 12
device = "cuda" if torch.cuda.is_available() else "cpu"

model = BabyDeepSeek(
    vocab_size=tokenizer.vocab_size,
    dim=256,
    layers=6
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Parameters: {total_params:,}"
)

print(
    f"Parameters: {total_params / 1e6:.2f}M"
)

In [ ]:
# @title Cell 13
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)

batch_size = 8

epochs = 3

print("Starting from scratch.")

In [ ]:
# @title Cell 14
model.train()

for epoch in range(epochs):

    total_loss = 0.0

    indices = list(range(len(dataset)))

    random.shuffle(indices)

    for start in range(
        0,
        len(indices),
        batch_size
    ):

        batch_ids = indices[
            start:start + batch_size
        ]

        input_ids = torch.stack([
            dataset[i]["input_ids"]
            for i in batch_ids
        ]).to(device)

        # Next-token prediction
        x = input_ids[:, :-1]
        y = input_ids[:, 1:]

        optimizer.zero_grad()

        logits, loss = model(
            x,
            y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        if start % (batch_size * 100) == 0:

            print(
                f"epoch={epoch+1} "
                f"step={start}/{len(indices)} "
                f"loss={loss.item():.4f}"
            )

    avg_loss = (
        total_loss /
        math.ceil(
            len(indices) / batch_size
        )
    )

    print(
        f"\nEpoch {epoch+1} "
        f"average loss = {avg_loss:.4f}\n"
    )

In [ ]:
# @title cell 16 Ask your New born AI
@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=50,
    temperature=0.8
):

    model.eval()

    ids = tokenizer(
        prompt,
        return_tensors="pt"
    )["input_ids"].to(device)

    for _ in range(max_new_tokens):

        logits, _ = model(ids)

        logits = logits[:, -1, :]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        ids = torch.cat(
            [ids, next_token],
            dim=1
        )

    return tokenizer.decode(
        ids[0],
        skip_special_tokens=True
    )


print(
    generate(
        model,
        "Once upon a time",
        max_new_tokens=60
    )
)

print("\n--- HELLO TEST ---")

print(
    generate(
        model,
        "hello",
        max_new_tokens=30
    )
)